# 🏥 Sistema de Gestión Hospitalaria (SGH)
### Curso de Programación Orientada a Objetos
**Ministerio de Salud y Protección Social — Colombia**

---

Presentado por: **MOISES DAVID BAQUERO DAZA y KEYNER STEVEN GARCIA ANAYA**



## CELDA 0 — Configuración de Google Drive y Rutas e Importación de Librerías

In [ ]:
import os
from enum import Enum
from datetime import datetime

# --- NUEVAS LIBRERÍAS AÑADIDAS ---
import re
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- MONTAJE DE GOOGLE DRIVE ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# =============================================================================
# CONFIGURACIÓN DE RUTAS DEL SISTEMA
# =============================================================================

# Carpeta principal
RUTA_BASE = '/content/drive/MyDrive/SGH/datos/'

# Crear carpeta automáticamente si no existe
os.makedirs(RUTA_BASE, exist_ok=True)

# Archivos del sistema
RUTA_PACIENTES      = os.path.join(RUTA_BASE, 'pacientes.txt')
RUTA_MEDICOS        = os.path.join(RUTA_BASE, 'medicos.txt')
RUTA_ESPECIALIDADES = os.path.join(RUTA_BASE, 'especialidades.txt')
RUTA_CITAS          = os.path.join(RUTA_BASE, 'citas.txt')
RUTA_CONSULTAS      = os.path.join(RUTA_BASE, 'consultas.txt')
RUTA_TRATAMIENTOS   = os.path.join(RUTA_BASE, 'tratamientos.txt')

print("✅ Rutas y librerías del sistema configuradas con éxito.")
print(f"📁 Ruta base: {RUTA_BASE}")

## SECCIÓN 1 — Modelo (Dominio)


### 1.1 — Enumeraciones del sistema

In [ ]:
from enum import Enum

class RegimenEnum(Enum):
    """Enum que representa los regímenes de aseguramiento en salud en Colombia."""
    BLANCO       = ' '
    CONTRIBUTIVO = 'CONTRIBUTIVO'
    SUBSIDIADO   = 'SUBSIDIADO'



class EstadoCitaEnum(Enum):
    """Enum que representa los posibles estados de una cita médica."""
    PROGRAMADA  = 'PROGRAMADA'
    ATENDIDA    = 'ATENDIDA'
    CANCELADA   = 'CANCELADA'
    NO_ASISTIO  = 'NO_ASISTIO'

print('Enumeraciones cargadas: RegimenEnum, EstadoCitaEnum')

### 1.2 — Clase Especialidad

In [ ]:
class Especialidad:
    """Representa una especialidad médica disponible en el hospital."""

    def __init__(self, codigo: str, nombre: str, descripcion: str):
        """
        Inicializa una especialidad médica.

        Args:
            codigo (str): Código único de la especialidad (ej: ESP-001).
            nombre (str): Nombre de la especialidad (ej: Cardiología).
            descripcion (str): Descripción breve de la especialidad.
        """
        self.codigo      = codigo
        self.nombre      = nombre
        self.descripcion = descripcion

    def to_linea(self) -> str:
        """Convierte el objeto a una línea de texto con campos separados por pipe."""
        return f'{self.codigo}|{self.nombre}|{self.descripcion}'

    @staticmethod
    def from_linea(linea: str) -> 'Especialidad':
        """Crea un objeto Especialidad a partir de una línea de texto del archivo."""
        campos = linea.strip().split('|')
        return Especialidad(campos[0], campos[1], campos[2])

    def __str__(self) -> str:
        return f'[{self.codigo}] {self.nombre} — {self.descripcion}'

print('Clase Especialidad cargada.')

### 1.3 — Clase Paciente

In [ ]:
class Paciente:
    """Representa un paciente registrado en el sistema hospitalario."""

    def __init__(self, num_documento: str, nombre: str, fecha_nacimiento: str,
                 tipo_sangre: str, eps: str, regimen: str, antecedentes: str):
        """
        Inicializa un paciente con sus datos personales y de aseguramiento.

        Args:
            num_documento (str): Número de documento de identidad del paciente.
            nombre (str): Nombre completo del paciente.
            fecha_nacimiento (str): Fecha de nacimiento en formato YYYY-MM-DD.
            tipo_sangre (str): Tipo de sangre (ej: O+, A-, B+).
            eps (str): Nombre de la EPS a la que pertenece.
            regimen (str): Régimen de afiliación (CONTRIBUTIVO (Contri), SUBSIDIADO (Subsi), etc.).
            antecedentes (str): Antecedentes médicos relevantes del paciente.
        """
        self.num_documento    = num_documento
        self.nombre           = nombre
        self.fecha_nacimiento = fecha_nacimiento
        self.tipo_sangre      = tipo_sangre
        self.eps              = eps
        self.regimen          = regimen
        self.antecedentes     = antecedentes

    def to_linea(self) -> str:
        """Convierte el objeto a una línea de texto con campos separados por pipe."""
        return (f'{self.num_documento}|{self.nombre}|{self.fecha_nacimiento}|'
                f'{self.tipo_sangre}|{self.eps}|{self.regimen}|{self.antecedentes}')

    @staticmethod
    def from_linea(linea: str) -> 'Paciente':
        """Crea un objeto Paciente a partir de una línea de texto del archivo."""
        campos = linea.strip().split('|')
        return Paciente(campos[0], campos[1], campos[2],
                        campos[3], campos[4], campos[5], campos[6])

    def __str__(self) -> str:
        return (f'Documento: {self.num_documento} | Nombre: {self.nombre} | '
                f'Nacimiento: {self.fecha_nacimiento} | Sangre: {self.tipo_sangre} | '
                f'EPS: {self.eps} | Régimen: {self.regimen} | '
                f'Antecedentes: {self.antecedentes}')

print('Clase Paciente cargada.')

### 1.4 — Clase Medico

In [ ]:
class Medico:
    """Representa un médico registrado en el sistema hospitalario."""

    def __init__(self, num_registro: str, nombre: str, especialidad: str,
                 consultorio: str, horario: str):
        """
        Inicializa un médico con sus datos profesionales.

        Args:
            num_registro (str): Número de registro médico único (ej: MED-001).
            nombre (str): Nombre completo del médico.
            especialidad (str): Código de la especialidad que ejerce.
            consultorio (str): Número o identificador del consultorio asignado.
            horario (str): Franja horaria de atención (ej: 07:00-13:00).
        """
        self.num_registro  = num_registro
        self.nombre        = nombre
        self.especialidad  = especialidad
        self.consultorio   = consultorio
        self.horario       = horario

    def to_linea(self) -> str:
        """Convierte el objeto a una línea de texto con campos separados por pipe."""
        return (f'{self.num_registro}|{self.nombre}|{self.especialidad}|'
                f'{self.consultorio}|{self.horario}')

    @staticmethod
    def from_linea(linea: str) -> 'Medico':
        """Crea un objeto Medico a partir de una línea de texto del archivo."""
        campos = linea.strip().split('|')
        return Medico(campos[0], campos[1], campos[2], campos[3], campos[4])

    def __str__(self) -> str:
        return (f'Registro: {self.num_registro} | Nombre: {self.nombre} | '
                f'Especialidad: {self.especialidad} | Consultorio: {self.consultorio} | '
                f'Horario: {self.horario}')

print('Clase Medico cargada.')

### 1.5 - Clase Cita

In [ ]:
class Cita:
    """Representa una cita médica programada en el sistema."""

    def __init__(self, codigo: str, doc_paciente: str, reg_medico: str, fecha: str, hora: str, motivo: str, estado: str):
        self.codigo = codigo
        self.doc_paciente = doc_paciente
        self.reg_medico = reg_medico
        self.fecha = fecha  # Formato DD-MM-YYYY
        self.hora = hora    # Formato HH:MM
        self.motivo = motivo
        self.estado = estado

    def to_linea(self) -> str:
        return f'{self.codigo}|{self.doc_paciente}|{self.reg_medico}|{self.fecha}|{self.hora}|{self.motivo}|{self.estado}'

    @staticmethod
    def from_linea(linea: str) -> 'Cita':
        campos = linea.strip().split('|')
        return Cita(campos[0], campos[1], campos[2], campos[3], campos[4], campos[5], campos[6])

    def __str__(self) -> str:
        return f'Cita: {self.codigo} | Paciente: {self.doc_paciente} | Médico: {self.reg_medico} | Fecha: {self.fecha} {self.hora} | Estado: {self.estado}'

print('Clase Cita cargada.')

### 1.6 - Clase Consulta y Clase Tratamiento

In [ ]:
class Consulta:
    """Representa el registro clínico de una consulta médica realizada."""

    def __init__(self, codigo_cita: str, diagnostico_desc: str, codigo_cie10: str, sintomas: str, presion: str, temperatura: str, frec_cardiaca: str, observaciones: str):
        self.codigo_cita = codigo_cita
        self.diagnostico_desc = diagnostico_desc
        self.codigo_cie10 = codigo_cie10
        self.sintomas = sintomas
        self.presion = presion
        self.temperatura = temperatura
        self.frec_cardiaca = frec_cardiaca
        self.observaciones = observaciones

    def to_linea(self) -> str:
        return f'{self.codigo_cita}|{self.diagnostico_desc}|{self.codigo_cie10}|{self.sintomas}|{self.presion}|{self.temperatura}|{self.frec_cardiaca}|{self.observaciones}'

    @staticmethod
    def from_linea(linea: str) -> 'Consulta':
        campos = linea.strip().split('|')
        return Consulta(campos[0], campos[1], campos[2], campos[3], campos[4], campos[5], campos[6], campos[7])

    def __str__(self) -> str:
        return f'Consulta Cita: {self.codigo_cita} | CIE-10: {self.codigo_cie10} | Diagnóstico: {self.diagnostico_desc} | Signos V.: P:{self.presion} T:{self.temperatura}°C FC:{self.frec_cardiaca} lpm'


class Tratamiento:
    """Representa un tratamiento o formulación asociado a una cita médica."""

    def __init__(self, codigo_cita: str, medicamento: str, dosis: str, frecuencia: str, duracion_dias: str):
        self.codigo_cita = codigo_cita
        self.medicamento = medicamento
        self.dosis = dosis
        self.frecuencia = frecuencia
        self.duracion_dias = duracion_dias

    def to_linea(self) -> str:
        return f'{self.codigo_cita}|{self.medicamento}|{self.dosis}|{self.frecuencia}|{self.duracion_dias}'

    @staticmethod
    def from_linea(linea: str) -> 'Tratamiento':
        campos = linea.strip().split('|')
        return Tratamiento(campos[0], campos[1], campos[2], campos[3], campos[4])

    def __str__(self) -> str:
        return f'Tratamiento Cita: {self.codigo_cita} | Med: {self.medicamento} | Dosis: {self.dosis} | Frec: {self.frecuencia} | Duración: {self.duracion_dias} días'

print('Clases de Registro Clínico cargadas.')

---
## SECCIÓN 2 — Repositorios


### 2.1 — ArchivoUtil

In [ ]:
class ArchivoUtil:
    """Utilidad estática para operaciones de lectura y escritura en archivos de texto."""

    @staticmethod
    def leer_lineas(ruta: str) -> list:
        """Lee todas las líneas no vacías de un archivo de texto."""
        try:
            with open(ruta, 'r', encoding='utf-8') as f:
                return [l.strip() for l in f if l.strip()]
        except FileNotFoundError:
            return []

    @staticmethod
    def escribir_linea(ruta: str, linea: str) -> None:
        """Agrega una línea al final del archivo (modo append)."""
        with open(ruta, 'a', encoding='utf-8') as f:
            f.write(linea + '\n')

    @staticmethod
    def reescribir(ruta: str, lineas: list) -> None:
        """Sobreescribe el archivo completo con la lista de líneas proporcionada."""
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lineas) + '\n')

    @staticmethod
    def generar_codigo(prefijo: str) -> str:
        """Genera un código único basado en el prefijo y la fecha/hora actual."""
        from datetime import datetime
        ts = datetime.now().strftime('%Y%m%d%H%M%S%f')[:17]
        return f'{prefijo}-{ts}'

print('Clase ArchivoUtil cargada.')

### 2.2 — EspecialidadRepository

In [ ]:
class EspecialidadRepository:
    """Repositorio para la persistencia de especialidades médicas en archivo de texto."""

    def __init__(self, ruta: str):
        """Inicializa el repositorio con la ruta del archivo de especialidades."""
        self.ruta = ruta

    def guardar(self, especialidad: Especialidad) -> None:
        """Persiste una nueva especialidad en el archivo."""
        ArchivoUtil.escribir_linea(self.ruta, especialidad.to_linea())

    def listar_todas(self) -> list:
        """Retorna una lista con todas las especialidades registradas."""
        lineas = ArchivoUtil.leer_lineas(self.ruta)
        return [Especialidad.from_linea(l) for l in lineas]

    def buscar_por_codigo(self, codigo: str):
        """Busca y retorna una especialidad por su código. Retorna None si no existe."""
        for esp in self.listar_todas():
            if esp.codigo == codigo:
                return esp
        return None

    def existe(self, codigo: str) -> bool:
        """Verifica si ya existe una especialidad con el código dado."""
        return self.buscar_por_codigo(codigo) is not None

print('Clase EspecialidadRepository cargada.')

### 2.3 — PacienteRepository

In [ ]:
class PacienteRepository:
    """Repositorio para la persistencia de pacientes en archivo de texto."""

    def __init__(self, ruta: str):
        """Inicializa el repositorio con la ruta del archivo de pacientes."""
        self.ruta = ruta

    def guardar(self, paciente: Paciente) -> None:
        """Persiste un nuevo paciente en el archivo."""
        ArchivoUtil.escribir_linea(self.ruta, paciente.to_linea())

    def listar_todos(self) -> list:
        """Retorna una lista con todos los pacientes registrados."""
        lineas = ArchivoUtil.leer_lineas(self.ruta)
        return [Paciente.from_linea(l) for l in lineas]

    def buscar_por_documento(self, num_documento: str):
        """Busca y retorna un paciente por número de documento. Retorna None si no existe."""
        for paciente in self.listar_todos():
            if paciente.num_documento == num_documento:
                return paciente
        return None

    def existe(self, num_documento: str) -> bool:
        """Verifica si ya existe un paciente con el número de documento dado."""
        return self.buscar_por_documento(num_documento) is not None

    def actualizar(self, paciente_actualizado: Paciente) -> bool:
        """Actualiza los datos de un paciente existente. Retorna True si fue exitoso."""
        lineas = ArchivoUtil.leer_lineas(self.ruta)
        nuevas_lineas = []
        encontrado = False
        for linea in lineas:
            campos = linea.split('|')
            if campos[0] == paciente_actualizado.num_documento:
                nuevas_lineas.append(paciente_actualizado.to_linea())
                encontrado = True
            else:
                nuevas_lineas.append(linea)
        if encontrado:
            ArchivoUtil.reescribir(self.ruta, nuevas_lineas)
        return encontrado

print('Clase PacienteRepository cargada.')

### 2.4 — MedicoRepository

In [ ]:
class MedicoRepository:
    """Repositorio para la persistencia de médicos en archivo de texto."""

    def __init__(self, ruta: str):
        """Inicializa el repositorio con la ruta del archivo de médicos."""
        self.ruta = ruta

    def guardar(self, medico: Medico) -> None:
        """Persiste un nuevo médico en el archivo."""
        ArchivoUtil.escribir_linea(self.ruta, medico.to_linea())

    def listar_todos(self) -> list:
        """Retorna una lista con todos los médicos registrados."""
        lineas = ArchivoUtil.leer_lineas(self.ruta)
        return [Medico.from_linea(l) for l in lineas]

    def buscar_por_registro(self, num_registro: str):
        """Busca y retorna un médico por número de registro. Retorna None si no existe."""
        for medico in self.listar_todos():
            if medico.num_registro == num_registro:
                return medico
        return None

    def existe(self, num_registro: str) -> bool:
        """Verifica si ya existe un médico con el número de registro dado."""
        return self.buscar_por_registro(num_registro) is not None

    def listar_por_especialidad(self, codigo_especialidad: str) -> list:
        """Retorna todos los médicos que pertenecen a una especialidad específica."""
        return [m for m in self.listar_todos() if m.especialidad == codigo_especialidad]

    def listar_por_horario(self, horario: str) -> list:
        """Retorna todos los médicos que atienden en una franja horaria específica."""
        return [m for m in self.listar_todos() if m.horario == horario]

print('Clase MedicoRepository cargada.')

### 2.5 - CitaRepository

In [ ]:
class CitaRepository:
    """Repositorio para la persistencia de citas médicas."""

    def __init__(self, ruta: str):
        self.ruta = ruta

    def guardar(self, cita: Cita) -> None:
        ArchivoUtil.escribir_linea(self.ruta, cita.to_linea())

    def listar_todas(self) -> list:
        lineas = ArchivoUtil.leer_lineas(self.ruta)
        return [Cita.from_linea(l) for l in lineas]

    def buscar_por_codigo(self, codigo: str):
        for cita in self.listar_todas():
            if cita.codigo == codigo:
                return cita
        return None

    def actualizar(self, cita_actualizada: Cita) -> bool:
        lineas = ArchivoUtil.leer_lineas(self.ruta)
        nuevas_lineas = []
        encontrado = False
        for linea in lineas:
            campos = linea.split('|')
            if campos[0] == cita_actualizada.codigo:
                nuevas_lineas.append(cita_actualizada.to_linea())
                encontrado = True
            else:
                nuevas_lineas.append(linea)
        if encontrado:
            ArchivoUtil.reescribir(self.ruta, nuevas_lineas)
        return encontrado

print('CitaRepository cargado.')

### 2.6 - RegistroClinicoRepository (Consultas y Tratamientos)

In [ ]:
class RegistroClinicoRepository:
    """Repositorio para guardar y listar consultas clínicas y tratamientos."""

    def __init__(self, ruta_consultas: str, ruta_tratamientos: str):
        self.ruta_consultas = ruta_consultas
        self.ruta_tratamientos = ruta_tratamientos

    def guardar_consulta(self, consulta: Consulta) -> None:
        ArchivoUtil.escribir_linea(self.ruta_consultas, consulta.to_linea())

    def guardar_tratamiento(self, tratamiento: Tratamiento) -> None:
        ArchivoUtil.escribir_linea(self.ruta_tratamientos, tratamiento.to_linea())

    def listar_consultas(self) -> list:
        lineas = ArchivoUtil.leer_lineas(self.ruta_consultas)
        return [Consulta.from_linea(l) for l in lineas]

    def listar_tratamientos(self) -> list:
        lineas = ArchivoUtil.leer_lineas(self.ruta_tratamientos)
        return [Tratamiento.from_linea(l) for l in lineas]

print('RegistroClinicoRepository cargado.')

---
## SECCIÓN 3 — Servicios


### Servicios

In [ ]:
class ValidadorData:
    """Clase estática encargada del cumplimiento estricto de formatos mediante expresiones regulares."""
    @staticmethod
    def validar_codigo_esp(texto: str) -> bool:
        # Valida que sea exactamente ESP- seguido de 3 números (Ej: ESP-001)
        return bool(re.match(r"^ESP-\d{3}$", texto))

    @staticmethod
    def validar_codigo_med(texto: str) -> bool:
        # Valida que sea exactamente MED- seguido de 3 números (Ej: MED-001)
        return bool(re.match(r"^MED-\d{3}$", texto))

    @staticmethod
    def validar_fecha(texto: str) -> bool:
        # Valida formato de fecha DD-MM-YYYY
        return bool(re.match(r"^\d{2}-\d{2}-\d{4}$", texto))

    @staticmethod
    def validar_hora(texto: str) -> bool:
        # Valida formato de hora militar HH:MM
        return bool(re.match(r"^\d{2}:\d{2}$", texto))

    @staticmethod
    def validar_presion(texto: str) -> bool:
        # Valida formato de presión arterial (Ej: 120/80)
        return bool(re.match(r"^\d{2,3}/\d{2,3}$", texto))

print("Validador de expresiones regulares cargado.")

### 3.1 — EspecialidadService

In [ ]:
class EspecialidadService:
    """Servicio con la lógica de negocio para la gestión de especialidades médicas."""

    def __init__(self, repositorio: EspecialidadRepository):
        """Inicializa el servicio con su repositorio de especialidades."""
        self.repositorio = repositorio

    def registrar(self, codigo: str, nombre: str, descripcion: str) -> str:
        """Registra una nueva especialidad médica. Retorna mensaje de resultado."""
        if self.repositorio.existe(codigo):
            return f'Ya existe una especialidad con el código {codigo}.'
        especialidad = Especialidad(codigo, nombre, descripcion)
        self.repositorio.guardar(especialidad)
        return f'Especialidad "{nombre}" registrada correctamente.'

    def consultar(self, codigo: str) -> str:
        """Consulta y retorna la información de una especialidad por código."""
        esp = self.repositorio.buscar_por_codigo(codigo)
        if esp:
            return str(esp)
        return f'No se encontró ninguna especialidad con el código {codigo}.'

    def listar_todas(self) -> str:
        """Lista todas las especialidades registradas en el sistema."""
        especialidades = self.repositorio.listar_todas()
        if not especialidades:
            return 'No hay especialidades registradas.'
        return '\n'.join([str(e) for e in especialidades])

print('Clase EspecialidadService cargada.')

### 3.2 — PacienteService

In [ ]:
class PacienteService:
    """Servicio con la lógica de negocio para la gestión de pacientes."""

    def __init__(self, repositorio: PacienteRepository):
        """Inicializa el servicio con su repositorio de pacientes."""
        self.repositorio = repositorio

    def registrar(self, num_documento: str, nombre: str, fecha_nacimiento: str,
                  tipo_sangre: str, eps: str, regimen: str, antecedentes: str) -> str:
        """Registra un nuevo paciente. Retorna mensaje de resultado."""
        if self.repositorio.existe(num_documento):
            return f'Ya existe un paciente con el documento {num_documento}.'
        regimen_upper = regimen.upper()
        valores_validos = [r.value for r in RegimenEnum]
        if regimen_upper not in valores_validos:
            return f'Régimen inválido. Opciones: {", ".join(valores_validos)}'
        paciente = Paciente(num_documento, nombre, fecha_nacimiento,
                            tipo_sangre, eps, regimen_upper, antecedentes)
        self.repositorio.guardar(paciente)
        return f'Paciente "{nombre}" registrado correctamente.'

    def consultar(self, num_documento: str) -> str:
        """Consulta y retorna la información de un paciente por número de documento."""
        paciente = self.repositorio.buscar_por_documento(num_documento)
        if paciente:
            return str(paciente)
        return f'No se encontró ningún paciente con el documento {num_documento}.'

    def actualizar(self, num_documento: str, nombre: str, fecha_nacimiento: str,
                   tipo_sangre: str, eps: str, regimen: str, antecedentes: str) -> str:
        """Actualiza los datos de un paciente existente. Retorna mensaje de resultado."""
        if not self.repositorio.existe(num_documento):
            return f'No existe un paciente con el documento {num_documento}.'
        regimen_upper = regimen.upper()
        valores_validos = [r.value for r in RegimenEnum]
        if regimen_upper not in valores_validos:
            return f'Régimen inválido. Opciones: {", ".join(valores_validos)}'
        paciente = Paciente(num_documento, nombre, fecha_nacimiento,
                            tipo_sangre, eps, regimen_upper, antecedentes)
        self.repositorio.actualizar(paciente)
        return f'Datos del paciente "{nombre}" actualizados correctamente.'

    def listar_todos(self) -> str:
        """Lista todos los pacientes registrados en el sistema."""
        pacientes = self.repositorio.listar_todos()
        if not pacientes:
            return 'No hay pacientes registrados.'
        resultado = f'Total de pacientes: {len(pacientes)}\n' + '-'*60 + '\n'
        resultado += '\n'.join([str(p) for p in pacientes])
        return resultado

print('Clase PacienteService cargada.')

### 3.3 — MedicoService

In [ ]:
class MedicoService:
    """Servicio con la lógica de negocio para la gestión del personal médico."""

    def __init__(self, repositorio: MedicoRepository,
                 repo_especialidades: EspecialidadRepository):
        """Inicializa el servicio con repositorios de médicos y especialidades."""
        self.repositorio         = repositorio
        self.repo_especialidades = repo_especialidades

    def registrar(self, num_registro: str, nombre: str, especialidad: str,
                  consultorio: str, horario: str) -> str:
        """Registra un nuevo médico. Retorna mensaje de resultado."""
        if self.repositorio.existe(num_registro):
            return f'Ya existe un médico con el registro {num_registro}.'
        if not self.repo_especialidades.existe(especialidad):
            return f'La especialidad con código "{especialidad}" no está registrada.'
        medico = Medico(num_registro, nombre, especialidad, consultorio, horario)
        self.repositorio.guardar(medico)
        return f'Médico "{nombre}" registrado correctamente.'

    def consultar(self, num_registro: str) -> str:
        """Consulta y retorna la información de un médico por número de registro."""
        medico = self.repositorio.buscar_por_registro(num_registro)
        if medico:
            return str(medico)
        return f'No se encontró ningún médico con el registro {num_registro}.'

    def listar_todos(self) -> str:
        """Lista todos los médicos registrados en el sistema."""
        medicos = self.repositorio.listar_todos()
        if not medicos:
            return 'No hay médicos registrados.'
        resultado = f'Total de médicos: {len(medicos)}\n' + '-'*60 + '\n'
        resultado += '\n'.join([str(m) for m in medicos])
        return resultado

    def listar_por_especialidad(self, codigo_especialidad: str) -> str:
        """Lista todos los médicos de una especialidad específica."""
        medicos = self.repositorio.listar_por_especialidad(codigo_especialidad)
        if not medicos:
            return f'No hay médicos registrados para la especialidad {codigo_especialidad}.'
        return '\n'.join([str(m) for m in medicos])

    def listar_por_horario(self, horario: str) -> str:
        """Lista todos los médicos que atienden en una franja horaria específica."""
        medicos = self.repositorio.listar_por_horario(horario)
        if not medicos:
            return f'No hay médicos con horario {horario}.'
        return '\n'.join([str(m) for m in medicos])

print('Clase MedicoService cargada.')

### 3.4 - CitaService

In [ ]:
class CitaService:
    """Lógica de negocio para agendamiento y cancelaciones de citas."""

    def __init__(self, repo_citas: CitaRepository, repo_pacientes: PacienteRepository, repo_medicos: MedicoRepository):
        self.repo_citas = repo_citas
        self.repo_pacientes = repo_pacientes
        self.repo_medicos = repo_medicos

    def programar_cita(self, doc_paciente: str, reg_medico: str, fecha: str, hora: str, motivo: str) -> str:
        if not self.repo_pacientes.existe(doc_paciente):
            return f'Error: El paciente con documento {doc_paciente} no existe.'
        if not self.repo_medicos.existe(reg_medico):
            return f'Error: El médico con registro {reg_medico} no existe.'

        # Validar cruce de horario simple (mismo médico, misma fecha y hora)
        for c in self.repo_citas.listar_todas():
            if c.reg_medico == reg_medico and c.fecha == fecha and c.hora == hora and c.estado == 'PROGRAMADA':
                return 'Error: El médico ya cuenta con una cita programada en esa fecha y hora.'

        codigo = ArchivoUtil.generar_codigo('CIT')
        nueva_cita = Cita(codigo, doc_paciente, reg_medico, fecha, hora, motivo, 'PROGRAMADA')
        self.repo_citas.guardar(nueva_cita)
        return f'Cita programada exitosamente con el código: {codigo}'

    def consultar_por_codigo(self, codigo: str) -> str:
        cita = self.repo_citas.buscar_por_codigo(codigo)
        return str(cita) if cita else f'No se encontró la cita con código {codigo}.'

    def cancelar_cita(self, codigo: str, motivo: str) -> str:
        cita = self.repo_citas.buscar_por_codigo(codigo)
        if not cita:
            return f'La cita {codigo} no existe.'
        if cita.estado != 'PROGRAMADA':
            return f'No se puede cancelar una cita con estado: {cita.estado}.'

        cita.estado  = 'CANCELADA'
        cita.motivo  = f'CANCELADA — {motivo}'          # guarda el motivo en el campo existente
        self.repo_citas.actualizar(cita)
        return f'La cita {codigo} ha sido cancelada. Motivo: {motivo}'

    def listar_por_filtro(self, tipo: str, identificador: str, f_inicio: str, f_fin: str) -> str:
        # Convertir fechas para comparar rangos cronológicos
        try:
            dt_inicio = datetime.strptime(f_inicio, '%d-%m-%Y')
            dt_fin = datetime.strptime(f_fin, '%d-%m-%Y')
        except ValueError:
            return "Formato de fechas incorrecto. Use DD-MM-YYYY."

        citas = self.repo_citas.listar_todas()
        filtradas = []
        for c in citas:
            try:
                c_fecha = datetime.strptime(c.fecha, '%d-%m-%Y')
                if dt_inicio <= c_fecha <= dt_fin:
                    if (tipo == 'PACIENTE' and c.doc_paciente == identificador) or (tipo == 'MEDICO' and c.reg_medico == identificador):
                        filtradas.append(c)
            except ValueError:
                continue

        if not filtradas:
            return 'No se encontraron citas en ese rango de fechas para el criterio seleccionado.'
        return '\n'.join([str(f) for f in filtradas])

print('Clase CitaService cargado.')

### 3.5 - RegistroClinicoService

In [ ]:
class RegistroClinicoService:
    """Lógica de negocio para el registro de la evolución clínica de pacientes."""

    def __init__(self, repo_clinico: RegistroClinicoRepository, repo_citas: CitaRepository):
        self.repo_clinico = repo_clinico
        self.repo_citas = repo_citas

    def registrar_consulta(self, codigo_cita: str, diagnostico: str, cie10: str, sintomas: str, presion: str, temperatura: str, frec_cardiaca: str, observaciones: str) -> str:
        cita = self.repo_citas.buscar_por_codigo(codigo_cita)
        if not cita:
            return f'Error: La cita médica {codigo_cita} no existe.'

        if cita.estado == EstadoCitaEnum.ATENDIDA.value:
            return f'Error: La cita {codigo_cita} ya cuenta con una evolución clínica registrada.'

        # Crear y persistir la consulta
        consulta = Consulta(codigo_cita, diagnostico, cie10, sintomas, presion, temperatura, frec_cardiaca, observaciones)
        self.repo_clinico.guardar_consulta(consulta)

        # Cambiar de forma estricta el estado de la cita a ATENDIDA
        cita.estado = EstadoCitaEnum.ATENDIDA.value
        self.repo_citas.actualizar(cita)

        return f'Evolución clínica guardada exitosamente y cita {codigo_cita} marcada como ATENDIDA.'

    def registrar_tratamiento(self, codigo_cita: str, medicamento: str, dosis: str, frecuencia: str, duracion: str) -> str:
        cita = self.repo_citas.buscar_por_codigo(codigo_cita)
        if not cita:
            return f'Error: La cita médica {codigo_cita} no existe.'

        if cita.estado != EstadoCitaEnum.ATENDIDA.value:
            return f'Error: No se puede añadir tratamiento. La cita {codigo_cita} debe estar en estado ATENDIDA (Evolución registrada).'

        # Crear instancia de tratamiento
        tratamiento = Tratamiento(codigo_cita, medicamento, dosis, frecuencia, duracion)

        # CORRECCIÓN DE LA LÍNEA: Cambiado de guardar_treatment a guardar_tratamiento
        self.repo_clinico.guardar_tratamiento(tratamiento)

        return f'Tratamiento añadido con éxito a la cita {codigo_cita}.'

    def consultar_historial_paciente(self, num_documento: str) -> dict:
        """Agrupa cronológicamente todas las consultas y tratamientos de un paciente específico."""
        citas_paciente = [c for c in self.repo_citas.listar_todas() if c.doc_paciente == num_documento]

        if not citas_paciente:
            return {"consultas": [], "tratamientos": []}

        codigos_citas = [c.codigo for c in citas_paciente]

        consultas_filtradas = [con for con in self.repo_clinico.listar_consultas() if con.codigo_cita in codigos_citas]
        tratamientos_filtrados = [tra for tra in self.repo_clinico.listar_tratamientos() if tra.codigo_cita in codigos_citas]

        return {
            "consultas": consultas_filtradas,
            "tratamientos": tratamientos_filtrados
        }

print('Clase RegistroClinicoService cargada y corregida.')

### 3.6 - AnalisisDatosService

In [ ]:
class AnalisisDatosService:
    """Implementa el módulo estadístico y reportes analíticos del hospital."""

    def __init__(self, repo_citas: CitaRepository, repo_clinico: RegistroClinicoRepository, repo_pacientes: PacienteRepository, repo_medicos: MedicoRepository):
        self.repo_citas = repo_citas
        self.repo_clinico = repo_clinico
        self.repo_pacientes = repo_pacientes
        self.repo_medicos = repo_medicos

    def calcular_promedio_consultas_por_medico(self, f_inicio: str, f_fin: str) -> str:
        try:
            dt_i = datetime.datetime.strptime(f_inicio, '%d-%m-%Y')
            dt_f = datetime.datetime.strptime(f_fin, '%d-%m-%Y')
        except ValueError:
            return "Formato incorrecto."

        citas = [c for c in self.repo_citas.listar_todas() if c.estado == 'ATENDIDA' and dt_i <= datetime.datetime.strptime(c.fecha, '%d-%m-%Y') <= dt_f]
        medicos = self.repo_medicos.listar_todos()

        if not medicos:
            return "No hay médicos registrados."

        promedio = len(citas) / len(medicos)
        return f'📊 Entre {f_inicio} y {f_fin}, se realizaron un promedio de {promedio:.2f} consultas por médico.'

    def listar_diagnosticos_frecuentes(self) -> str:
        consultas = self.repo_clinico.listar_consultas()
        if not consultas:
            return "No se registran diagnósticos en el sistema."

        frecuencias = {}
        for c in consultas:
            frecuencias[c.codigo_cie10] = frecuencias.get(c.codigo_cie10, 0) + 1

        # Ordenar de mayor a menor frecuencia
        ranking = sorted(frecuencias.items(), key=lambda x: x[1], reverse=True)
        res = "🏆 RANKING DE DIAGNÓSTICOS MÁS FRECUENTES (CIE-10):\n"
        for cie, cant in ranking:
            res += f'  • Código {cie}: {cant} caso(s)\n'
        return res

    def calcular_tasa_ocupacion_especialidad(self) -> str:
        citas = [c for c in self.repo_citas.listar_todas() if c.estado == 'PROGRAMADA']
        medicos = self.repo_medicos.listar_todos()

        if not citas:
            return "No hay citas actualmente programadas para medir ocupación."

        conteo_esp = {}
        for c in citas:
            med = self.repo_medicos.buscar_por_registro(c.reg_medico)
            if med:
                conteo_esp[med.especialidad] = conteo_esp.get(med.especialidad, 0) + 1

        res = "📈 TASAS DE OCUPACIÓN ABSOLUTA POR ESPECIALIDAD:\n"
        for esp, cant in conteo_esp.items():
            res += f'  • Especialidad {esp}: {cant} cita(s) asignada(s)\n'
        return res

    def generar_reporte_distribucion_pacientes(self) -> str:
        pacientes = self.repo_pacientes.listar_todos()
        if not pacientes:
            return "No hay pacientes en la base de datos."

        dist = {}
        for p in pacientes:
            clave = f'EPS: {p.eps} | Régimen: {p.regimen}'
            dist[clave] = dist.get(clave, 0) + 1

        res = "📋 REPORTE DE AFILIACIÓN (DISTRIBUCIÓN DE PACIENTES):\n"
        for k, v in dist.items():
            res += f'  • {k} -> {v} paciente(s)\n'
        return res

    def listar_diagnosticos_frecuentes(self, n: int = 5) -> str:
        consultas = self.repo_clinico.listar_consultas()
        if not consultas:
            return "No se registran diagnósticos en el sistema."
        frecuencias = {}
        for c in consultas:
            frecuencias[c.codigo_cie10] = frecuencias.get(c.codigo_cie10, 0) + 1
        ranking = sorted(frecuencias.items(), key=lambda x: x[1], reverse=True)[:n]
        res = f"🏆 TOP {n} DIAGNÓSTICOS MÁS FRECUENTES (CIE-10):\n"
        for cie, cant in ranking:
            res += f'  • Código {cie}: {cant} caso(s)\n'
        return res

    def identificar_medicamentos_mas_prescritos(self, n: int = 5) -> str:
        trats = self.repo_clinico.listar_tratamientos()
        if not trats:
            return "No se han formulado medicamentos aún."
        conteo_med = {}
        for t in trats:
            nombre = t.medicamento.upper().strip()
            conteo_med[nombre] = conteo_med.get(nombre, 0) + 1
        ranking = sorted(conteo_med.items(), key=lambda x: x[1], reverse=True)[:n]
        res = f"💊 TOP {n} MEDICAMENTOS MÁS PRESCRITOS:\n"
        for med, cant in ranking:
            res += f'  • {med}: Formulada {cant} vez/veces.\n'
        return res

print('Clase AnalisisDatosService cargado.')


### 3.7 - Dependencias

In [ ]:
# =====================================================================
# INYECCIÓN DE DEPENDENCIAS: CREACIÓN REAL DE LOS OBJETOS DE SERVICIO
# ====================================================================

# 1. Inicialización de las bases físicas de datos (Repositorios)
repo_especialidades = EspecialidadRepository(RUTA_ESPECIALIDADES)
repo_pacientes      = PacienteRepository(RUTA_PACIENTES)
repo_medicos        = MedicoRepository(RUTA_MEDICOS)
repo_citas          = CitaRepository(RUTA_CITAS)
repo_clinico        = RegistroClinicoRepository(RUTA_CONSULTAS, RUTA_TRATAMIENTOS)

# 2. Construcción de los servicios del ecosistema hospitalario (AQUÍ SE CREAN)
svc_especialidades = EspecialidadService(repo_especialidades)
svc_pacientes      = PacienteService(repo_pacientes)
svc_medicos        = MedicoService(repo_medicos, repo_especialidades)
svc_citas          = CitaService(repo_citas, repo_pacientes, repo_medicos)
svc_clinico        = RegistroClinicoService(repo_clinico, repo_citas)
svc_analisis       = AnalisisDatosService(repo_citas, repo_clinico, repo_pacientes, repo_medicos)

print("Todos los servicios instanciados y acoplados con éxito. Memoria lista.")

---
## SECCIÓN 4 — Interfaz de Usuario


### Interfaz con HTML y CSS

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import datetime

# =============================================================================
# LAYOUT ESTÁNDAR
# =============================================================================
LABEL_W = '175px'
TOTAL_W = '540px'
STYLE   = {'description_width': LABEL_W}

def lyt():
    return widgets.Layout(width=TOTAL_W)

def lyt_btn(w='180px'):
    return widgets.Layout(width=w, margin='4px 4px 4px 0px')

MSG_OBLIGATORIOS = "⚠️ Todos los campos son obligatorios. Por favor completa la información antes de continuar."

# =============================================================================
# CSS
# =============================================================================
CSS = widgets.HTML("""
<style>
.sgh-banner {
    background: linear-gradient(135deg, #005c53 0%, #007a6e 100%);
    color: white; padding: 20px; text-align: center;
    border-radius: 10px; font-family: Arial, sans-serif; margin-bottom: 14px;
}
.sgh-banner h1 { margin: 0 0 6px 0; font-size: 1.5em; }
.sgh-banner p  { margin: 0; font-size: 0.95em; opacity: 0.9; }
.sgh-sub {
    border-left: 5px solid #9fc131; padding: 4px 0 4px 12px;
    margin: 16px 0 10px 0; color: #c8f0eb;
    font-weight: bold; font-size: 1.05em; font-family: Arial, sans-serif;
}
.sgh-ok {
    background: #dbf5f0; color: #00423c; padding: 10px 14px;
    border-radius: 6px; border: 1px solid #00796b;
    font-family: Arial, sans-serif; font-size: 13px; margin-top: 8px;
}
.sgh-err {
    background: #fce8e6; color: #a51d24; padding: 10px 14px;
    border-radius: 6px; border: 1px solid #c62828;
    font-family: Arial, sans-serif; font-size: 13px; margin-top: 8px;
}
.sgh-table {
    border-collapse: collapse; width: 100%;
    font-family: Arial, sans-serif; font-size: 13px; margin-top: 6px;
}
.sgh-table thead tr { background-color: #005c53; color: #ffffff; }
.sgh-table th {
    padding: 9px 14px; text-align: left; font-weight: bold;
    border: 1px solid #004a42; white-space: nowrap;
}
.sgh-table td {
    padding: 7px 14px; border: 1px solid #b2dfdb;
    color: #1a1a1a; white-space: nowrap;
}
.sgh-table tbody tr:nth-child(even) { background: #f0f9f7; }
.sgh-table tbody tr:nth-child(odd)  { background: #ffffff; }
.sgh-table tbody tr:hover           { background: #c8ede8; }
.sgh-wrap { overflow-x: auto; margin-top: 8px; border-radius: 6px; }
.sgh-total {
    color: #005c53; font-family: Arial, sans-serif;
    font-size: 13px; font-weight: bold; margin: 8px 0 2px 0;
}
</style>
""")

# =============================================================================
# HELPERS
# =============================================================================
def tabla_html(headers, filas):
    ths = "".join(f"<th>{h}</th>" for h in headers)
    trs = "".join(
        "<tr>" + "".join(f"<td>{v if v is not None else ''}</td>" for v in f) + "</tr>"
        for f in filas
    )
    return (
        f"<div class='sgh-wrap'><table class='sgh-table'>"
        f"<thead><tr>{ths}</tr></thead><tbody>{trs}</tbody>"
        f"</table></div>"
    )

def mostrar_tabla(out_widget, headers, filas, entidad="registro"):
    """Actualiza el Output con la tabla, sin colapsar el panel."""
    html = (
        f"<p class='sgh-total'>Total: {len(filas)} {entidad}(s)</p>"
        + tabla_html(headers, filas)
    )
    out_widget.clear_output(wait=True)
    with out_widget:
        display(HTML(html))

def mostrar_ok(out_widget, msg):
    out_widget.clear_output(wait=True)
    with out_widget:
        display(HTML(f"<div class='sgh-ok'>✅ {msg}</div>"))

def mostrar_err(out_widget, msg):
    out_widget.clear_output(wait=True)
    with out_widget:
        display(HTML(f"<div class='sgh-err'>❌ {msg}</div>"))

def sub(texto):
    return widgets.HTML(f"<div class='sgh-sub'>{texto}</div>")

def get_msg(res):
    return res if isinstance(res, str) else getattr(res, 'mensaje', str(res))

def exito(msg):
    return any(p in msg.lower() for p in [
        "correctamente","exitosamente","registrada","registrado",
        "actualizado","incorporado","programada","cancelada","añadido","éxito"
    ])

# =============================================================================
# HORARIOS
# =============================================================================
horas_disponibles = []
_t = datetime.time(6, 0)
while True:
    horas_disponibles.append(_t.strftime("%H:%M"))
    _h = _t.hour + ((_t.minute + 30) // 60)
    _m = (_t.minute + 30) % 60
    if _h > 19: break
    _t = datetime.time(_h, _m)

horarios_medicos = [
    "06:00-06:30","06:30-07:00","07:00-07:30","07:30-08:00",
    "08:00-08:30","08:30-09:00","09:00-09:30","09:30-10:00",
    "10:00-10:30","10:30-11:00","11:00-11:30","11:30-12:00",
    "12:00-12:30","12:30-13:00","13:00-13:30","13:30-14:00",
    "14:00-14:30","14:30-15:00","15:00-15:30","15:30-16:00",
    "16:00-16:30","16:30-17:00","17:00-17:30","17:30-18:00",
    "18:00-18:30","18:30-19:00","19:00-19:30","19:30-20:00"
]

# =============================================================================
# PESTAÑA 1 — ESPECIALIDADES
# =============================================================================
txt_esp_cod  = widgets.Text(description="Código:",       placeholder="ESP-001", layout=lyt(), style=STYLE)
txt_esp_nom  = widgets.Text(description="Nombre:",       placeholder="Ej: Cardiología", layout=lyt(), style=STYLE)
txt_esp_desc = widgets.Text(description="Descripción:",  placeholder="Descripción breve de la especialidad", layout=lyt(), style=STYLE)
btn_esp_reg  = widgets.Button(description="Registrar",    button_style="success", icon="save",  layout=lyt_btn())
btn_esp_lst  = widgets.Button(description="Listar Todas", button_style="info",    icon="list",  layout=lyt_btn())
out_esp      = widgets.Output()

def limpiar_esp():
    txt_esp_cod.value = txt_esp_nom.value = txt_esp_desc.value = ""

def reg_esp_click(b):
    cod, nom, desc = txt_esp_cod.value.strip(), txt_esp_nom.value.strip(), txt_esp_desc.value.strip()
    if not cod or not nom or not desc:
        mostrar_err(out_esp, MSG_OBLIGATORIOS); return
    res = svc_especialidades.registrar(cod, nom, desc)
    msg = get_msg(res)
    if exito(msg):
        limpiar_esp(); mostrar_ok(out_esp, msg)
    else:
        mostrar_err(out_esp, msg)

def lst_esp_click(b):
    lista = repo_especialidades.listar_todas()
    if not lista:
        mostrar_err(out_esp, "No hay especialidades registradas."); return
    mostrar_tabla(out_esp,
        ["Código", "Nombre", "Descripción"],
        [(e.codigo, e.nombre, e.descripcion) for e in lista],
        "especialidad"
    )

btn_esp_reg.on_click(reg_esp_click)
btn_esp_lst.on_click(lst_esp_click)

box_esp = widgets.VBox([
    sub("Gestión de Especialidades Médicas"),
    txt_esp_cod, txt_esp_nom, txt_esp_desc,
    widgets.HBox([btn_esp_reg, btn_esp_lst]),
    out_esp
])

# =============================================================================
# PESTAÑA 2 — PACIENTES
# =============================================================================
txt_pac_doc   = widgets.Text(description="Número documento:", placeholder="Solo números", layout=lyt(), style=STYLE)
txt_pac_nom   = widgets.Text(description="Nombre completo:",  placeholder="Ej: Juan Pérez López", layout=lyt(), style=STYLE)
date_pac_fec  = widgets.DatePicker(description="Fecha nacimiento:", layout=lyt(), style=STYLE)
drop_pac_sang = widgets.Dropdown(options=["A+","A-","B+","B-","AB+","AB-","O+","O-"],
                                 description="Tipo de sangre:", layout=lyt(), style=STYLE)
txt_pac_eps   = widgets.Text(description="EPS:", placeholder="Nombre de la EPS", layout=lyt(), style=STYLE)
drop_pac_reg  = widgets.Dropdown(options=[r.value for r in RegimenEnum],
                                 description="Régimen:", layout=lyt(), style=STYLE)
txt_pac_ante  = widgets.Text(description="Antecedentes médicos:", placeholder="Antecedentes del paciente", layout=lyt(), style=STYLE)
btn_pac_reg   = widgets.Button(description="Registrar Paciente",  button_style="success", icon="user-plus", layout=lyt_btn('200px'))
btn_pac_act   = widgets.Button(description="Actualizar Paciente", button_style="warning",  icon="edit",     layout=lyt_btn('200px'))
btn_pac_lst   = widgets.Button(description="Listar Todos",        button_style="info",     icon="users",    layout=lyt_btn())
out_pac       = widgets.Output()

def limpiar_pac():
    txt_pac_doc.value = txt_pac_nom.value = txt_pac_eps.value = txt_pac_ante.value = ""
    date_pac_fec.value = None

def validar_pac():
    if not all([txt_pac_doc.value.strip(), txt_pac_nom.value.strip(),
                txt_pac_eps.value.strip(), txt_pac_ante.value.strip(),
                date_pac_fec.value]):
        return False, MSG_OBLIGATORIOS
    if not txt_pac_doc.value.strip().isdigit():
        return False, "El número de documento debe contener únicamente dígitos."
    return True, ""

def reg_pac_click(b):
    v, e = validar_pac()
    if not v: mostrar_err(out_pac, e); return
    res = svc_pacientes.registrar(
        txt_pac_doc.value.strip(), txt_pac_nom.value.strip(),
        date_pac_fec.value.strftime("%d-%m-%Y"),
        drop_pac_sang.value, txt_pac_eps.value.strip(),
        drop_pac_reg.value, txt_pac_ante.value.strip()
    )
    msg = get_msg(res)
    if exito(msg): limpiar_pac(); mostrar_ok(out_pac, msg)
    else: mostrar_err(out_pac, msg)

def act_pac_click(b):
    v, e = validar_pac()
    if not v: mostrar_err(out_pac, e); return
    res = svc_pacientes.actualizar(
        txt_pac_doc.value.strip(), txt_pac_nom.value.strip(),
        date_pac_fec.value.strftime("%d-%m-%Y"),
        drop_pac_sang.value, txt_pac_eps.value.strip(),
        drop_pac_reg.value, txt_pac_ante.value.strip()
    )
    msg = get_msg(res)
    if exito(msg): limpiar_pac(); mostrar_ok(out_pac, msg)
    else: mostrar_err(out_pac, msg)

def lst_pac_click(b):
    lista = repo_pacientes.listar_todos()
    if not lista:
        mostrar_err(out_pac, "No hay pacientes registrados."); return
    mostrar_tabla(out_pac,
        ["Documento","Nombre","Nacimiento","Sangre","EPS","Régimen","Antecedentes"],
        [(p.num_documento, p.nombre, p.fecha_nacimiento,
          p.tipo_sangre, p.eps, p.regimen, p.antecedentes) for p in lista],
        "paciente"
    )

btn_pac_reg.on_click(reg_pac_click)
btn_pac_act.on_click(act_pac_click)
btn_pac_lst.on_click(lst_pac_click)

box_pac = widgets.VBox([
    sub("Admisión e Historias de Pacientes"),
    txt_pac_doc, txt_pac_nom, date_pac_fec, drop_pac_sang,
    txt_pac_eps, drop_pac_reg, txt_pac_ante,
    widgets.HBox([btn_pac_reg, btn_pac_act, btn_pac_lst]),
    out_pac
])

# =============================================================================
# PESTAÑA 3 — MÉDICOS
# =============================================================================
txt_med_reg  = widgets.Text(description="Registro médico:",      placeholder="MED-001", layout=lyt(), style=STYLE)
txt_med_nom  = widgets.Text(description="Nombre completo:",      placeholder="Ej: Dr. Luis Gómez", layout=lyt(), style=STYLE)
txt_med_esp  = widgets.Text(description="Código especialidad:",  placeholder="ESP-001", layout=lyt(), style=STYLE)
txt_med_cons = widgets.Text(description="Número consultorio:",   placeholder="Solo números", layout=lyt(), style=STYLE)
drop_med_hor = widgets.Dropdown(options=horarios_medicos, description="Horario:", layout=lyt(), style=STYLE)
btn_med_reg  = widgets.Button(description="Registrar Médico", button_style="success", icon="user-md", layout=lyt_btn('180px'))
btn_med_lst  = widgets.Button(description="Listar Planta",    button_style="info",    icon="list",    layout=lyt_btn())
out_med      = widgets.Output()

def limpiar_med():
    txt_med_reg.value = txt_med_nom.value = txt_med_esp.value = txt_med_cons.value = ""

def validar_med():
    if not all([txt_med_reg.value.strip(), txt_med_nom.value.strip(),
                txt_med_esp.value.strip(), txt_med_cons.value.strip()]):
        return False, MSG_OBLIGATORIOS
    if not txt_med_cons.value.strip().isdigit():
        return False, "El número de consultorio debe contener únicamente dígitos."
    return True, ""

def reg_med_click(b):
    v, e = validar_med()
    if not v: mostrar_err(out_med, e); return
    res = svc_medicos.registrar(
        txt_med_reg.value.strip(), txt_med_nom.value.strip(),
        txt_med_esp.value.strip(), txt_med_cons.value.strip(), drop_med_hor.value
    )
    msg = get_msg(res)
    if exito(msg): limpiar_med(); mostrar_ok(out_med, msg)
    else: mostrar_err(out_med, msg)

def lst_med_click(b):
    lista = repo_medicos.listar_todos()
    if not lista:
        mostrar_err(out_med, "No hay médicos registrados."); return
    mostrar_tabla(out_med,
        ["Registro","Nombre","Especialidad","Consultorio","Horario"],
        [(m.num_registro, m.nombre, m.especialidad, m.consultorio, m.horario) for m in lista],
        "médico"
    )

drop_med_hor_fil = widgets.Dropdown(
    options=horarios_medicos,
    description="Horario a buscar:",
    layout=lyt(), style=STYLE
)
btn_med_hor_bus  = widgets.Button(
    description="Buscar por Horario",
    button_style="warning", icon="clock",
    layout=lyt_btn('200px')
)
out_med_hor      = widgets.Output()

def bus_med_hor_click(b):
    horario = drop_med_hor_fil.value
    lista   = repo_medicos.listar_por_horario(horario)
    if not lista:
        mostrar_err(out_med_hor, f"No hay médicos con horario {horario}."); return
    mostrar_tabla(out_med_hor,
        ["Registro", "Nombre", "Especialidad", "Consultorio", "Horario"],
        [(m.num_registro, m.nombre, m.especialidad, m.consultorio, m.horario) for m in lista],
        "médico"
    )

btn_med_hor_bus.on_click(bus_med_hor_click)

btn_med_reg.on_click(reg_med_click)
btn_med_lst.on_click(lst_med_click)

box_med = widgets.VBox([
    sub("Control del Personal Médico"),
    txt_med_reg, txt_med_nom, txt_med_esp, txt_med_cons, drop_med_hor,
    widgets.HBox([btn_med_reg, btn_med_lst]),
    out_med,
    sub("Buscar Médicos por Franja Horaria"),
    drop_med_hor_fil,
    widgets.HBox([btn_med_hor_bus]),
    out_med_hor,
   ])

# =============================================================================
# PESTAÑA 4 — CITAS
# =============================================================================

# -- Helpers para dropdowns dinámicos --
def _opts_pacientes():
    lista = repo_pacientes.listar_todos()
    base  = [("— Seleccione un paciente —", "")]
    if not lista: return base
    return base + [(f"{p.num_documento}  |  {p.nombre}", p.num_documento) for p in lista]

def _opts_medicos():
    lista = repo_medicos.listar_todos()
    base  = [("— Seleccione un médico —", "")]
    if not lista: return base
    return base + [(f"{m.num_registro}  |  {m.nombre}", m.num_registro) for m in lista]

def _opts_citas_prog():
    citas = repo_citas.listar_todas()
    prog  = [c for c in citas if c.estado == "PROGRAMADA"]
    base  = [("— Seleccione una cita —", "")]
    if not prog: return base
    return base + [(f"{c.codigo}  |  {c.doc_paciente}  |  {c.fecha} {c.hora}", c.codigo) for c in prog]

def _opts_citas_ate():
    citas = repo_citas.listar_todas()
    ate   = [c for c in citas if c.estado == "ATENDIDA"]
    base  = [("— Seleccione cita atendida —", "")]
    if not ate: return base
    return base + [(f"{c.codigo}  |  {c.doc_paciente}  |  {c.fecha}", c.codigo) for c in ate]

# Widgets
txt_cit_id   = widgets.Text(description="ID de la cita:", placeholder="CIT-001", layout=lyt(), style=STYLE)
drop_cit_pac = widgets.Dropdown(options=_opts_pacientes(), description="Paciente:", layout=lyt(), style=STYLE)
drop_cit_med = widgets.Dropdown(options=_opts_medicos(),   description="Médico:",   layout=lyt(), style=STYLE)
date_cit_fec = widgets.DatePicker(description="Fecha de la cita:", layout=lyt(), style=STYLE)
drop_cit_hor = widgets.Dropdown(options=horas_disponibles, description="Hora:", layout=lyt(), style=STYLE)
txt_cit_mot  = widgets.Text(description="Motivo consulta:", placeholder="Motivo de la cita", layout=lyt(), style=STYLE)

btn_cit_ref  = widgets.Button(description="↻ Actualizar listas", button_style="",       icon="refresh",        layout=lyt_btn('180px'))
btn_cit_reg  = widgets.Button(description="Agendar Cita",         button_style="primary",icon="calendar-check", layout=lyt_btn('180px'))
btn_cit_lst  = widgets.Button(description="Ver Agenda",            button_style="info",   icon="list",           layout=lyt_btn())
out_cit_form = widgets.Output()   # mensajes de registro

drop_cit_can = widgets.Dropdown(options=_opts_citas_prog(), description="Cita a cancelar:", layout=lyt(), style=STYLE)
txt_cit_mot_can = widgets.Text(
    description="Motivo cancelación:",
    placeholder="Razón por la que se cancela la cita",
    layout=lyt(), style=STYLE
)
btn_cit_ref2 = widgets.Button(description="↻ Actualizar lista", button_style="", icon="refresh",       layout=lyt_btn('180px'))
btn_cit_can  = widgets.Button(description="Cancelar Turno",      button_style="danger", icon="calendar-times", layout=lyt_btn('180px'))
out_cit_can  = widgets.Output()   # mensajes de cancelación

out_cit_lst  = widgets.Output()   # tabla agenda

def limpiar_cit():
    txt_cit_id.value       = ""
    drop_cit_pac.options   = _opts_pacientes()
    drop_cit_pac.value     = ""
    drop_cit_med.options   = _opts_medicos()
    drop_cit_med.value     = ""
    date_cit_fec.value     = None
    drop_cit_hor.value     = horas_disponibles[0]
    txt_cit_mot.value      = ""

def validar_cit():
    if not all([txt_cit_id.value.strip(), drop_cit_pac.value,
                drop_cit_med.value, txt_cit_mot.value.strip(),
                date_cit_fec.value]):
        return False, MSG_OBLIGATORIOS
    return True, ""

def ref_cit_click(b):
    drop_cit_pac.options = _opts_pacientes()
    drop_cit_med.options = _opts_medicos()

def reg_cit_click(b):
    v, e = validar_cit()
    if not v: mostrar_err(out_cit_form, e); return
    res = svc_citas.programar_cita(
        drop_cit_pac.value, drop_cit_med.value,
        date_cit_fec.value.strftime("%d-%m-%Y"),
        drop_cit_hor.value, txt_cit_mot.value.strip()
    )
    msg = get_msg(res)
    if exito(msg):
        limpiar_cit()
        drop_cit_can.options = _opts_citas_prog()
        mostrar_ok(out_cit_form, msg)
    else:
        mostrar_err(out_cit_form, msg)

def lst_cit_click(b):
    lista = repo_citas.listar_todas()
    if not lista:
        mostrar_err(out_cit_lst, "No hay citas registradas."); return
    mostrar_tabla(out_cit_lst,
        ["ID Cita","Doc. Paciente","Reg. Médico","Fecha","Hora","Motivo","Estado"],
        [(c.codigo, c.doc_paciente, c.reg_medico,
          c.fecha, c.hora, c.motivo, c.estado) for c in lista],
        "cita"
    )

def ref_cit2_click(b):
    drop_cit_can.options = _opts_citas_prog()

def can_cit_click(b):
    codigo = drop_cit_can.value
    motivo = txt_cit_mot_can.value.strip()
    if not codigo:
        mostrar_err(out_cit_can, "Selecciona una cita de la lista para cancelar."); return
    if not motivo:
        mostrar_err(out_cit_can, "Debes ingresar el motivo de cancelación."); return
    res = svc_citas.cancelar_cita(codigo, motivo)
    msg = get_msg(res)
    if exito(msg):
        txt_cit_mot_can.value    = ''
        drop_cit_can.options     = _opts_citas_prog()
        mostrar_ok(out_cit_can, msg)
    else:
        mostrar_err(out_cit_can, msg)

# --- Widgets para búsqueda por rango de fechas ---
drop_cit_tipo   = widgets.Dropdown(
    options=[('Paciente', 'PACIENTE'), ('Médico', 'MEDICO')],
    description="Buscar por:",
    layout=lyt(), style=STYLE
)
txt_cit_id_fil  = widgets.Text(
    description="ID (doc/registro):",
    placeholder="Número de documento o registro médico",
    layout=lyt(), style=STYLE
)
date_cit_ini    = widgets.DatePicker(description="Fecha inicio:", layout=lyt(), style=STYLE)
date_cit_fin    = widgets.DatePicker(description="Fecha fin:",    layout=lyt(), style=STYLE)
btn_cit_buscar  = widgets.Button(
    description="Buscar Citas",
    button_style="primary", icon="search",
    layout=lyt_btn('160px')
)
out_cit_buscar  = widgets.Output()

def buscar_cit_click(b):
    tipo  = drop_cit_tipo.value
    ident = txt_cit_id_fil.value.strip()
    if not ident or not date_cit_ini.value or not date_cit_fin.value:
        mostrar_err(out_cit_buscar, "Completa el ID y las dos fechas para buscar."); return
    f_ini = date_cit_ini.value.strftime('%d-%m-%Y')
    f_fin = date_cit_fin.value.strftime('%d-%m-%Y')
    res   = svc_citas.listar_por_filtro(tipo, ident, f_ini, f_fin)
    msg   = get_msg(res)
    # Si devuelve texto plano con citas, lo mostramos como tabla
    if 'No se encontraron' in msg or 'incorrecto' in msg:
        mostrar_err(out_cit_buscar, msg); return
    # Parsear los objetos directamente para tabla
    try:
        dt_i = __import__('datetime').datetime.strptime(f_ini, '%d-%m-%Y')
        dt_f = __import__('datetime').datetime.strptime(f_fin, '%d-%m-%Y')
        todas = repo_citas.listar_todas()
        filtradas = [
            c for c in todas
            if dt_i <= __import__('datetime').datetime.strptime(c.fecha, '%d-%m-%Y') <= dt_f
            and ((tipo == 'PACIENTE' and c.doc_paciente == ident)
                 or (tipo == 'MEDICO' and c.reg_medico == ident))
        ]
        if not filtradas:
            mostrar_err(out_cit_buscar, "No se encontraron citas en ese rango."); return
        mostrar_tabla(out_cit_buscar,
            ["ID Cita", "Doc. Paciente", "Reg. Médico", "Fecha", "Hora", "Motivo", "Estado"],
            [(c.codigo, c.doc_paciente, c.reg_medico,
              c.fecha, c.hora, c.motivo, c.estado) for c in filtradas],
            "cita"
        )
    except Exception as e:
        mostrar_err(out_cit_buscar, f"Error al filtrar: {e}")

btn_cit_buscar.on_click(buscar_cit_click)

btn_cit_ref.on_click(ref_cit_click)
btn_cit_reg.on_click(reg_cit_click)
btn_cit_lst.on_click(lst_cit_click)
btn_cit_ref2.on_click(ref_cit2_click)
btn_cit_can.on_click(can_cit_click)

box_cit = widgets.VBox([
    sub("Agendamiento de Citas"),
    txt_cit_id, drop_cit_pac, drop_cit_med, date_cit_fec, drop_cit_hor, txt_cit_mot,
    widgets.HBox([btn_cit_ref, btn_cit_reg, btn_cit_lst]),
    out_cit_form,
    out_cit_lst,
    sub("Cancelación de Turnos"),
    drop_cit_can,
    txt_cit_mot_can,
    widgets.HBox([btn_cit_ref2, btn_cit_can]),

    sub("Buscar Citas por Rango de Fechas"),
    drop_cit_tipo,
    txt_cit_id_fil,
    date_cit_ini,
    date_cit_fin,
    widgets.HBox([btn_cit_buscar]),
    out_cit_buscar,

])

# =============================================================================
# PESTAÑA 5 — EVOLUCIÓN CLÍNICA
# =============================================================================
drop_cli_cod = widgets.Dropdown(options=_opts_citas_prog(), description="Cita a atender:", layout=lyt(), style=STYLE)
txt_cli_diag = widgets.Text(description="Diagnóstico:",       placeholder="Descripción del diagnóstico", layout=lyt(), style=STYLE)
txt_cli_cie  = widgets.Text(description="Código CIE-10:",     placeholder="Ej: J06.9",   layout=lyt(), style=STYLE)
txt_cli_sint = widgets.Text(description="Síntomas:",          placeholder="Síntomas del paciente", layout=lyt(), style=STYLE)
txt_cli_pres = widgets.Text(description="Presión arterial:",  placeholder="Ej: 120/80",  layout=lyt(), style=STYLE)
txt_cli_temp = widgets.Text(description="Temperatura (°C):",  placeholder="Ej: 36.5",    layout=lyt(), style=STYLE)
txt_cli_fc   = widgets.Text(description="Frec. cardíaca:",    placeholder="Ej: 75 bpm",  layout=lyt(), style=STYLE)
txt_cli_obs  = widgets.Text(description="Observaciones:",     placeholder="Evolución clínica del paciente", layout=lyt(), style=STYLE)
btn_cli_ref  = widgets.Button(description="↻ Actualizar lista",  button_style="",       icon="refresh",   layout=lyt_btn('180px'))
btn_cli_reg  = widgets.Button(description="Registrar Consulta",  button_style="success",icon="heartbeat", layout=lyt_btn('200px'))
btn_cli_lst  = widgets.Button(description="Listar Consultas",    button_style="info",   icon="list",      layout=lyt_btn())
out_cli_form = widgets.Output()
out_cli_lst  = widgets.Output()

drop_tra_cod = widgets.Dropdown(options=_opts_citas_ate(), description="Cita atendida:", layout=lyt(), style=STYLE)
txt_tra_med  = widgets.Text(description="Medicamento:",      placeholder="Nombre del medicamento", layout=lyt(), style=STYLE)
txt_tra_dos  = widgets.Text(description="Dosis:",            placeholder="Ej: 500mg",        layout=lyt(), style=STYLE)
txt_tra_fre  = widgets.Text(description="Frecuencia:",       placeholder="Ej: Cada 8 horas", layout=lyt(), style=STYLE)
txt_tra_dur  = widgets.Text(description="Duración (días):",  placeholder="Solo números",     layout=lyt(), style=STYLE)
btn_tra_ref  = widgets.Button(description="↻ Actualizar lista",   button_style="",      icon="refresh", layout=lyt_btn('180px'))
btn_tra_reg  = widgets.Button(description="Añadir Tratamiento",   button_style="warning",icon="pills",  layout=lyt_btn('200px'))
btn_tra_lst  = widgets.Button(description="Listar Tratamientos",  button_style="info",  icon="list",    layout=lyt_btn())
out_tra_form = widgets.Output()
out_tra_lst  = widgets.Output()

def limpiar_cli():
    drop_cli_cod.options = _opts_citas_prog()
    drop_cli_cod.value   = ""
    for w in [txt_cli_diag, txt_cli_cie, txt_cli_sint,
              txt_cli_pres, txt_cli_temp, txt_cli_fc, txt_cli_obs]:
        w.value = ""

def limpiar_tra():
    drop_tra_cod.options = _opts_citas_ate()
    drop_tra_cod.value   = ""
    for w in [txt_tra_med, txt_tra_dos, txt_tra_fre, txt_tra_dur]:
        w.value = ""

def validar_cli():
    campos = [txt_cli_diag, txt_cli_cie, txt_cli_sint,
              txt_cli_pres, txt_cli_temp, txt_cli_fc, txt_cli_obs]
    if not drop_cli_cod.value or any(not w.value.strip() for w in campos):
        return False, MSG_OBLIGATORIOS
    return True, ""

def validar_tra():
    campos = [txt_tra_med, txt_tra_dos, txt_tra_fre, txt_tra_dur]
    if not drop_tra_cod.value or any(not w.value.strip() for w in campos):
        return False, MSG_OBLIGATORIOS
    if not txt_tra_dur.value.strip().isdigit():
        return False, "La duración debe ser un número entero de días."
    return True, ""

def ref_cli_click(b):
    drop_cli_cod.options = _opts_citas_prog()

def reg_cli_click(b):
    v, e = validar_cli()
    if not v: mostrar_err(out_cli_form, e); return
    res = svc_clinico.registrar_consulta(
        drop_cli_cod.value, txt_cli_diag.value.strip(),
        txt_cli_cie.value.strip(), txt_cli_sint.value.strip(),
        txt_cli_pres.value.strip(), txt_cli_temp.value.strip(),
        txt_cli_fc.value.strip(), txt_cli_obs.value.strip()
    )
    msg = get_msg(res)
    if exito(msg):
        limpiar_cli()
        drop_tra_cod.options = _opts_citas_ate()
        mostrar_ok(out_cli_form, msg)
    else:
        mostrar_err(out_cli_form, msg)

def lst_cli_click(b):
    lista = repo_clinico.listar_consultas()
    if not lista:
        mostrar_err(out_cli_lst, "No hay consultas registradas."); return
    mostrar_tabla(out_cli_lst,
        ["ID Cita","Diagnóstico","CIE-10","Síntomas","P. Arterial","Temp °C","Frec. Cardíaca","Observaciones"],
        [(c.codigo_cita, c.diagnostico_desc, c.codigo_cie10, c.sintomas,
          c.presion, c.temperatura, c.frec_cardiaca, c.observaciones) for c in lista],
        "consulta"
    )

def ref_tra_click(b):
    drop_tra_cod.options = _opts_citas_ate()

def reg_tra_click(b):
    v, e = validar_tra()
    if not v: mostrar_err(out_tra_form, e); return
    res = svc_clinico.registrar_tratamiento(
        drop_tra_cod.value, txt_tra_med.value.strip(),
        txt_tra_dos.value.strip(), txt_tra_fre.value.strip(),
        txt_tra_dur.value.strip()
    )
    msg = get_msg(res)
    if exito(msg): limpiar_tra(); mostrar_ok(out_tra_form, msg)
    else: mostrar_err(out_tra_form, msg)

def lst_tra_click(b):
    lista = repo_clinico.listar_tratamientos()
    if not lista:
        mostrar_err(out_tra_lst, "No hay tratamientos registrados."); return
    mostrar_tabla(out_tra_lst,
        ["ID Cita","Medicamento","Dosis","Frecuencia","Duración (días)"],
        [(t.codigo_cita, t.medicamento, t.dosis, t.frecuencia, t.duracion_dias) for t in lista],
        "tratamiento"
    )

txt_hist_doc  = widgets.Text(
    description="Documento paciente:",
    placeholder="Número de documento",
    layout=lyt(), style=STYLE
)
btn_hist_ver  = widgets.Button(
    description="Ver Historial",
    button_style="info", icon="history",
    layout=lyt_btn('160px')
)
out_hist      = widgets.Output()

def ver_historial_click(b):
    doc = txt_hist_doc.value.strip()
    if not doc:
        mostrar_err(out_hist, "Ingresa el número de documento del paciente."); return
    # Verificar que el paciente existe
    if not repo_pacientes.existe(doc):
        mostrar_err(out_hist, f"No existe un paciente con documento {doc}."); return
    historial = svc_clinico.consultar_historial_paciente(doc)
    consultas    = historial.get('consultas', [])
    tratamientos = historial.get('tratamientos', [])
    if not consultas and not tratamientos:
        mostrar_err(out_hist, "Este paciente no tiene consultas registradas aún."); return
    out_hist.clear_output(wait=True)
    with out_hist:
        from IPython.display import display, HTML
        html = f"<p class='sgh-total'>Historial clínico — Documento: {doc}</p>"
        if consultas:
            html += "<p class='sgh-total'>Consultas</p>"
            html += tabla_html(
                ["ID Cita", "Diagnóstico", "CIE-10", "Síntomas",
                 "P. Arterial", "Temp °C", "Frec. Cardíaca", "Observaciones"],
                [(c.codigo_cita, c.diagnostico_desc, c.codigo_cie10, c.sintomas,
                  c.presion, c.temperatura, c.frec_cardiaca, c.observaciones)
                 for c in consultas]
            )
        if tratamientos:
            html += "<p class='sgh-total'>Tratamientos</p>"
            html += tabla_html(
                ["ID Cita", "Medicamento", "Dosis", "Frecuencia", "Duración (días)"],
                [(t.codigo_cita, t.medicamento, t.dosis, t.frecuencia, t.duracion_dias)
                 for t in tratamientos]
            )
        display(HTML(html))

btn_hist_ver.on_click(ver_historial_click)

btn_cli_ref.on_click(ref_cli_click)
btn_cli_reg.on_click(reg_cli_click)
btn_cli_lst.on_click(lst_cli_click)
btn_tra_ref.on_click(ref_tra_click)
btn_tra_reg.on_click(reg_tra_click)
btn_tra_lst.on_click(lst_tra_click)

box_cli = widgets.VBox([
    sub("Registro de Consulta Médica"),
    drop_cli_cod, txt_cli_diag, txt_cli_cie, txt_cli_sint,
    txt_cli_pres, txt_cli_temp, txt_cli_fc, txt_cli_obs,
    widgets.HBox([btn_cli_ref, btn_cli_reg, btn_cli_lst]),
    out_cli_form, out_cli_lst,
    sub("Formulación de Tratamiento"),
    drop_tra_cod, txt_tra_med, txt_tra_dos, txt_tra_fre, txt_tra_dur,
    widgets.HBox([btn_tra_ref, btn_tra_reg, btn_tra_lst]),
    out_tra_form, out_tra_lst,
    sub("Historial Clínico del Paciente"),
    txt_hist_doc,
    widgets.HBox([btn_hist_ver]),
    out_hist,

])

# =============================================================================
# PESTAÑA 6 — MÓDULO ANALÍTICO
# =============================================================================
# --- Widgets de control ---
date_ana_ini   = widgets.DatePicker(description="Fecha inicio:", layout=lyt(), style=STYLE)
date_ana_fin   = widgets.DatePicker(description="Fecha fin:",    layout=lyt(), style=STYLE)
btn_ana_prom   = widgets.Button(
    description="Ver Promedio Consultas",
    button_style="primary", icon="chart-line",
    layout=lyt_btn('220px')
)
txt_ana_n_diag = widgets.BoundedIntText(
    value=5, min=1, max=50,
    description="Top N diagnósticos:",
    layout=lyt(), style=STYLE
)
btn_ana_diag   = widgets.Button(
    description="Ver Diagnósticos",
    button_style="info", icon="stethoscope",
    layout=lyt_btn('200px')
)
txt_ana_n_med  = widgets.BoundedIntText(
    value=5, min=1, max=50,
    description="Top N medicamentos:",
    layout=lyt(), style=STYLE
)
btn_ana_med    = widgets.Button(
    description="Ver Medicamentos",
    button_style="warning", icon="pills",
    layout=lyt_btn('200px')
)
btn_ana_rep    = widgets.Button(
    description="Auditoría General",
    button_style="danger", icon="chart-pie",
    layout=widgets.Layout(width='220px', margin='6px 0')
)
out_ana        = widgets.Output()

# --- Handlers ---
def ana_prom_click(b):
    if not date_ana_ini.value or not date_ana_fin.value:
        mostrar_err(out_ana, "Selecciona fecha inicio y fecha fin."); return
    f_ini = date_ana_ini.value.strftime('%d-%m-%Y')
    f_fin = date_ana_fin.value.strftime('%d-%m-%Y')
    res   = svc_analisis.calcular_promedio_consultas_por_medico(f_ini, f_fin)
    mostrar_ok(out_ana, get_msg(res))

def ana_diag_click(b):
    n   = txt_ana_n_diag.value
    res = svc_analisis.listar_diagnosticos_frecuentes(n)
    out_ana.clear_output(wait=True)
    with out_ana:
        display(HTML(f"<div class='sgh-ok'><pre style='margin:0'>{res}</pre></div>"))

def ana_med_click(b):
    n   = txt_ana_n_med.value
    res = svc_analisis.identificar_medicamentos_mas_prescritos(n)
    out_ana.clear_output(wait=True)
    with out_ana:
        display(HTML(f"<div class='sgh-ok'><pre style='margin:0'>{res}</pre></div>"))

def rep_click(b):
    pacientes    = repo_pacientes.listar_todos()
    medicos      = repo_medicos.listar_todos()
    citas        = repo_citas.listar_todas()
    consultas    = repo_clinico.listar_consultas()
    tratamientos = repo_clinico.listar_tratamientos()

    resumen = [
        ("👤 Pacientes",    len(pacientes)),
        ("🩺 Médicos",      len(medicos)),
        ("📅 Citas",        len(citas)),
        ("📋 Consultas",    len(consultas)),
        ("💊 Tratamientos", len(tratamientos)),
    ]
    html = "<p class='sgh-total'>📊 Resumen General del Sistema</p>"
    html += tabla_html(["Entidad", "Total Registros"], resumen)

    if pacientes:
        dist = {}
        for p in pacientes:
            k = f"{p.eps} | {p.regimen}"
            dist[k] = dist.get(k, 0) + 1
        html += "<p class='sgh-total'>📋 Distribución por EPS / Régimen</p>"
        html += tabla_html(["EPS | Régimen", "N° Pacientes"], list(dist.items()))

    # Tasa de ocupación por especialidad (citas ATENDIDAS / total programadas)
    citas_atendidas  = [c for c in citas if c.estado == 'ATENDIDA']
    citas_programadas = [c for c in citas if c.estado in ('PROGRAMADA', 'ATENDIDA')]
    if citas_programadas:
        esp_total = {}
        esp_ate   = {}
        for c in citas_programadas:
            med = repo_medicos.buscar_por_registro(c.reg_medico)
            if med:
                esp_total[med.especialidad] = esp_total.get(med.especialidad, 0) + 1
        for c in citas_atendidas:
            med = repo_medicos.buscar_por_registro(c.reg_medico)
            if med:
                esp_ate[med.especialidad]   = esp_ate.get(med.especialidad, 0) + 1
        filas_esp = [
            (esp, esp_total[esp], esp_ate.get(esp, 0),
             f"{100 * esp_ate.get(esp, 0) / esp_total[esp]:.1f}%")
            for esp in esp_total
        ]
        html += "<p class='sgh-total'>📈 Tasa de Ocupación por Especialidad</p>"
        html += tabla_html(
            ["Especialidad", "Citas programadas", "Citas atendidas", "Tasa (%)"],
            filas_esp
        )

    out_ana.clear_output(wait=True)
    with out_ana:
        display(HTML(html))

btn_ana_prom.on_click(ana_prom_click)
btn_ana_diag.on_click(ana_diag_click)
btn_ana_med.on_click(ana_med_click)
btn_ana_rep.on_click(rep_click)

box_ana = widgets.VBox([
    sub("Promedio de Consultas por Médico"),
    date_ana_ini,
    date_ana_fin,
    widgets.HBox([btn_ana_prom]),
    sub("Diagnósticos más Frecuentes (CIE-10)"),
    txt_ana_n_diag,
    widgets.HBox([btn_ana_diag]),
    sub("Medicamentos más Prescritos"),
    txt_ana_n_med,
    widgets.HBox([btn_ana_med]),
    sub("Auditoría General del Sistema"),
    widgets.HBox([btn_ana_rep]),
    out_ana,
])

# =============================================================================
# BOTÓN GLOBAL PARA REGRESAR / CERRAR VISTAS DE DATOS
# =============================================================================
btn_limpiar_global = widgets.Button(
    description="Regresar / Cerrar Datos",
    button_style="danger",  # Color rojo estandarizado para cierres
    icon="arrow-left",
    layout=widgets.Layout(width='220px', margin='10px 0px 0px auto')  # Alineado a la derecha
)

def limpiar_pantalla_global(b):
    """Limpia de manera simultánea todos los paneles de salida (Output) del sistema."""
    outputs_sistema = [
        out_esp, out_pac, out_med, out_cit_form,
        out_cit_lst, out_cit_can, out_cli_form,
        out_cli_lst, out_tra_form, out_tra_lst, out_ana
    ]
    for salida in outputs_sistema:
        salida.clear_output()

btn_limpiar_global.on_click(limpiar_pantalla_global)

# =============================================================================
# MONTAJE FINAL
# =============================================================================
tabs = widgets.Tab(children=[box_esp, box_pac, box_med, box_cit, box_cli, box_ana])
for i, t in enumerate(["Especialidades","Pacientes","Médicos","Citas Médicas","Evolución Clínica","Métricas"]):
    tabs.set_title(i, t)

banner = widgets.HTML("""
<div class='sgh-banner'>
    <h1>🏥 SISTEMA DE GESTIÓN HOSPITALARIA (SGH)</h1>
    <p>Módulo Médico e Historias Clínicas Electrónicas</p>
</div>
""")

# Fila horizontal que empuja el botón al extremo inferior derecho
box_inferior_derecho = widgets.HBox(
    [btn_limpiar_global],
    layout=widgets.Layout(width='100%', justify_content='flex-end')
)



---
## SECCIÓN 5 — Ejecución Principal

In [ ]:
# Renderizado estructurado
interfaz_completa = widgets.VBox([
    CSS,
    banner,
    tabs,
    box_inferior_derecho
])

display(interfaz_completa)